In [1]:
!pip install transformers[torch] datasets scikit-learn matplotlib seaborn

In [3]:
import pandas as pd
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Pastikan menggunakan GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan device: {device.upper()}")

# Konfigurasi Eksperimen
apps = ['satusehat', 'signal', 'ikd']
models_to_test = {
    'IndoBERT': 'indobenchmark/indobert-base-p1',
    'mBERT': 'bert-base-multilingual-uncased'
}

hasil_evaluasi = []

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

for app in apps:
    print(f"\n{'='*50}\nMEMPROSES APLIKASI: {app.upper()}\n{'='*50}")

    # 1. Load Data
    path_file = f"/content/data/{app}_cleaned_reviews.csv"
    df = pd.read_csv(path_file)

    df['label'] = df['sentiment'].apply(lambda x: 1 if x == 'Positif' else 0)
    df = df.dropna(subset=['cleaned_content'])

    # Split Data
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        df['cleaned_content'].tolist(),
        df['label'].tolist(),
        test_size=0.2,
        random_state=42,
        stratify=df['label'].tolist()
    )

    for model_name, model_path in models_to_test.items():
        print(f"\n---> Mulai Fine-Tuning: {model_name} pada {app.upper()} <---")

        # 2. Load Tokenizer & Model
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2).to(device)

        train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
        test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

        class SentDataset(torch.utils.data.Dataset):
            def __init__(self, encodings, labels):
                self.encodings = encodings
                self.labels = labels
            def __getitem__(self, idx):
                item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
                item['labels'] = torch.tensor(self.labels[idx])
                return item
            def __len__(self):
                return len(self.labels)

        train_dataset = SentDataset(train_encodings, train_labels)
        test_dataset = SentDataset(test_encodings, test_labels)

        # 3. Konfigurasi Training (ERROR SUDAH DIPERBAIKI DI SINI)
        training_args = TrainingArguments(
            output_dir=f'./results_{model_name}_{app}',
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            eval_strategy="epoch",  # <-- INI YANG DIUBAH
            save_strategy="epoch",
            logging_dir='./logs',
            load_best_model_at_end=True,
            fp16=True
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics
        )

        # 4. Training
        trainer.train()

        # 5. Evaluasi
        eval_result = trainer.evaluate()
        print(f"Hasil {model_name} - {app.upper()}: {eval_result}")

        hasil_evaluasi.append({
            'Aplikasi': app.upper(),
            'Model': model_name,
            'Accuracy': round(eval_result['eval_accuracy'] * 100, 2),
            'Precision': round(eval_result['eval_precision'] * 100, 2),
            'Recall': round(eval_result['eval_recall'] * 100, 2),
            'F1-Score': round(eval_result['eval_f1'] * 100, 2)
        })

        # 6. Confusion Matrix
        predictions = trainer.predict(test_dataset)
        preds = predictions.predictions.argmax(-1)
        cm = confusion_matrix(test_labels, preds)

        # --- TAMBAHAN 1: SIMPAN PREDIKSI UNTUK UJI STATISTIK ---
        if 'prediksi_sementara' not in locals():
            prediksi_sementara = {}
        prediksi_sementara[model_name] = preds
        # -------------------------------------------------------

        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negatif', 'Positif'], yticklabels=['Negatif', 'Positif'])
        plt.xlabel('Tebakan Model (Predicted)')
        plt.ylabel('Kenyataan (Actual)')
        plt.title(f'Confusion Matrix: {model_name} - {app.upper()}')

        nama_gambar_cm = f"CM_{model_name}_{app}.png"
        plt.savefig(nama_gambar_cm, dpi=300, bbox_inches='tight')
        plt.close()

        # --- TAMBAHAN 2: UJI MCNEMAR DAN MACRO F1 (Dijalankan per Aplikasi) ---
    from sklearn.metrics import f1_score
    from statsmodels.stats.contingency_tables import mcnemar

    print(f"\n{'*'*50}")
    print(f"🔍 ANALISIS STATISTIK UNTUK APLIKASI: {app.upper()}")
    print(f"{'*'*50}")

    y_true = np.array(test_labels)
    y_pred_indo = prediksi_sementara['IndoBERT']
    y_pred_mbert = prediksi_sementara['mBERT']

    # Hitung Macro F1
    macro_f1_indo = f1_score(y_true, y_pred_indo, average='macro')
    macro_f1_mbert = f1_score(y_true, y_pred_mbert, average='macro')

    print(f"IndoBERT (Macro F1) : {macro_f1_indo * 100:.2f}%")
    print(f"mBERT    (Macro F1) : {macro_f1_mbert * 100:.2f}%\n")

    # Uji McNemar
    indo_correct = (y_true == y_pred_indo)
    mbert_correct = (y_true == y_pred_mbert)

    table = [
        [np.sum(indo_correct & mbert_correct), np.sum(indo_correct & ~mbert_correct)],
        [np.sum(~indo_correct & mbert_correct), np.sum(~indo_correct & ~mbert_correct)]
    ]

    result = mcnemar(table, exact=False, correction=True)

    print(f"P-value Uji McNemar : {result.pvalue:.4f}")
    if result.pvalue < 0.05:
        print("KESIMPULAN: Perbedaan SIGNIFIKAN secara statistik! (IndoBERT bukan sekadar menang untung-untungan).")
    else:
        print("KESIMPULAN: Perbedaan TIDAK signifikan.")

    # Hapus prediksi sementara untuk aplikasi berikutnya
    del prediksi_sementara
    # ----------------------------------------------------------------------

# 7. Simpan Hasil Akhir
df_hasil = pd.DataFrame(hasil_evaluasi)
df_hasil.to_excel('Tabel_Performa_Benchmarking.xlsx', index=False)
print("\n" + "="*50)
print("🎉🎉🎉 SELURUH PROSES SELESAI! 🎉🎉🎉")

Menggunakan device: CUDA

MEMPROSES APLIKASI: SATUSEHAT

---> Mulai Fine-Tuning: IndoBERT pada SATUSEHAT <---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Hasil mBERT - IKD: {'eval_loss': 0.29570868611335754, 'eval_accuracy': 0.8973544973544973, 'eval_f1': 0.8908988053642143, 'eval_precision': 0.9052668918104274, 'eval_recall': 0.8973544973544973, 'eval_runtime': 1.627, 'eval_samples_per_second': 580.817, 'eval_steps_per_second': 18.439, 'epoch': 3.0}

**************************************************
🔍 ANALISIS STATISTIK UNTUK APLIKASI: IKD
**************************************************
IndoBERT (Macro F1) : 89.94%
mBERT    (Macro F1) : 86.04%

P-value Uji McNemar : 0.0021
KESIMPULAN: Perbedaan SIGNIFIKAN secara statistik! (IndoBERT bukan sekadar menang untung-untungan).

🎉🎉🎉 SELURUH PROSES SELESAI! 🎉🎉🎉


In [5]:
from sklearn.metrics import f1_score
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

# 1. Menghitung Macro F1-Score (Menjawab kritik Prof soal Class Imbalance)
macro_f1_indo = f1_score(y_true, y_pred_indo, average='macro')
macro_f1_mbert = f1_score(y_true, y_pred_mbert, average='macro')

print("=== PERBANDINGAN MACRO F1-SCORE ===")
print(f"IndoBERT (Macro F1): {macro_f1_indo:.4f}")
print(f"mBERT    (Macro F1): {macro_f1_mbert:.4f}\n")

# 2. Uji Signifikansi McNemar (Menjawab kritik Prof soal signifikansi statistik)
# Menentukan record mana yang benar/salah untuk tiap model
indo_correct = (y_true == y_pred_indo)
mbert_correct = (y_true == y_pred_mbert)

# Membuat tabel kontingensi 2x2
# [ [Keduanya Benar, Indo Benar & mBERT Salah],
#   [Indo Salah & mBERT Benar, Keduanya Salah] ]
table = [[np.sum(indo_correct & mbert_correct), np.sum(indo_correct & ~mbert_correct)],
         [np.sum(~indo_correct & mbert_correct), np.sum(~indo_correct & ~mbert_correct)]]

result = mcnemar(table, exact=False, correction=True)

print("=== HASIL UJI MCNEMAR ===")
print(f"Statistic: {result.statistic:.3f}")
print(f"P-value: {result.pvalue:.4f}")

if result.pvalue < 0.05:
    print("KESIMPULAN: Perbedaan performa antara IndoBERT dan mBERT signifikan secara statistik.")
else:
    print("KESIMPULAN: Perbedaan performa tidak signifikan secara statistik.")

=== PERBANDINGAN MACRO F1-SCORE ===
IndoBERT (Macro F1): 0.8994
mBERT    (Macro F1): 0.8604

=== HASIL UJI MCNEMAR ===
Statistic: 9.490
P-value: 0.0021
KESIMPULAN: Perbedaan performa antara IndoBERT dan mBERT signifikan secara statistik.
